In [1]:
import pandas as pd
import numpy as np
from pgmpy.base.DAG import DAG
from pgmpy.estimators.CITests import chi_square
from pgmpy.independencies import IndependenceAssertion


In [2]:

# Build the causal DAG.
G = DAG()
G.add_edges_from(
    [
      ('A','E'),
      ('S','E'),
      ('E','O'),
      ('E','R'),
      ('O','T'),
      ('R','T')
    ]
)

# Load N examples from the data
survey_url = "https://raw.githubusercontent.com/altdeep/causalML/master/datasets/transportation_survey.csv"

N = 30
full_data = pd.read_csv(survey_url)
data = full_data[:N]

# List D-Separations
dseps = G.get_independencies()
print(dseps)


(T ⟂ A | O, R)
(S ⟂ T | O, R)
(A ⟂ O | E)
(R ⟂ O | E)
(S ⟂ O | E)
(S ⟂ R | E)
(A ⟂ R | E)
(S ⟂ A)
(E ⟂ T | O, R)


In [16]:

# Run Chi-squared tests for independence
significance = .01

def test_dsep(dsep: IndependenceAssertion, data = data, significance=significance):
  test_outputs = []
  for X in list(dsep.get_assertion()[0]):
    for Y in list(dsep.get_assertion()[1]):
      Z = list(dsep.get_assertion()[2])
      test_result = chi_square(X=X, Y=Y, Z=Z, data=data, boolean=True, significance_level=significance)
      test_outputs.append((IndependenceAssertion(X, Y, Z), test_result))
  return test_outputs

results = [test_dsep(dsep, data) for dsep in dseps.get_assertions()]
results_flat = [item for sublist in results for item in sublist]
results = {k: v for k, v in results_flat}
print(results)


{(T ⟂ A | O, R): np.True_, (S ⟂ T | O, R): np.True_, (A ⟂ O | E): np.True_, (R ⟂ O | E): np.False_, (S ⟂ O | E): np.True_, (S ⟂ R | E): np.True_, (A ⟂ R | E): np.True_, (S ⟂ A): np.True_, (E ⟂ T | O, R): np.True_}


In [11]:

# Hint on how to count the number of Trues.
sum(results.values())

np.int64(8)

## increase N

In [6]:
len(full_data)

500

With more data, more tests pass

In [ ]:
def count_true(dseps, N, significance=significance):
    data = full_data[:N]
    results = [test_dsep(dsep, data, significance) for dsep in dseps.get_assertions()]
    results_flat = [item for sublist in results for item in sublist]
    results = {k: v for k, v in results_flat}
    return sum(results.values()) 

[ count_true(dseps, N) for N in [30,  100, 500] ]

[np.int64(8), np.int64(9), np.int64(9)]

As significance goes down, MORE tests pass.

In [21]:
[count_true(dseps, 30, significance=0.9), count_true(dseps, 30, significance=0.05)]

[np.int64(3), np.int64(8)]